# Explore Indexed Chunks

This notebook allows you to explore the chunks that have been indexed in your contextual retrieval system.

In [4]:
import os
import sys
from pathlib import Path

# Add backend to path
backend_path = Path.cwd().parent.parent / "backend" / "src"
sys.path.insert(0, str(backend_path))

# Set environment variables
os.environ["DJANGO_SETTINGS_MODULE"] = "uu_backend.django_project.settings.local"
os.environ["DJANGO_DATABASE_URL"] = "postgres://uu:uu@localhost:5432/uu_django"
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

print("✓ Django setup complete")

✓ Django setup complete


In [5]:
from uu_backend.django_data.models import DocumentModel
from uu_backend.services.contextual_retrieval.service import ContextualRetrievalService
from uu_backend.services.contextual_retrieval.vector_store import ChromaVectorStore
import pandas as pd

print("✓ Imports complete")

✓ Imports complete


## 1. List Available Documents

In [14]:
# Get all documents with completed indexing
docs = DocumentModel.objects.filter(retrieval_index_status='completed').order_by('-created_at')

doc_data = []
for doc in docs:
    doc_data.append({
        'ID': str(doc.id),
        'Filename': doc.filename,
        'Chunks': doc.retrieval_chunks_count,
        'Created': doc.created_at.strftime('%Y-%m-%d %H:%M')
    })

df_docs = pd.DataFrame(doc_data)
print(f"Found {len(df_docs)} indexed documents:\n")
df_docs

Found 1 indexed documents:



,ID,Filename,Chunks,Created
0,8269b380-d311-472e-b5ed-d3e4476ba880,ACORD 25 fillable-test-filled-v3.pdf,2,2026-02-20 03:41


## 2. View Chunks from a Specific Document

Copy a document ID from above and paste it below:

In [ ]:
# Fetch chunks via API (easier than direct ChromaDB access)
import requests

API_URL = "http://localhost:8000/api/v1"

def get_document_chunks(document_id):
    """Get chunks for a document via API."""
    response = requests.get(f"{API_URL}/retrieval/documents/{document_id}/chunks")
    response.raise_for_status()
    return response.json()

# Select a document ID from the list above
document_id = "8269b380-d311-472e-b5ed-d3e4476ba880"  # Replace with your document ID

try:
    data = get_document_chunks(document_id)
    chunks = data.get("chunks", [])
    
    chunks_data = []
    for i, chunk in enumerate(chunks):
        chunks_data.append({
            'Index': i,
            'Chunk ID': chunk.get('chunk_id', 'N/A'),
            'Content Preview': chunk['text'][:100] + '...' if len(chunk['text']) > 100 else chunk['text'],
            'Full Content': chunk['text'],
            'Context': chunk.get('context', 'N/A'),
            'Page': chunk.get('metadata', {}).get('page_number', 'N/A'),
            'Length': len(chunk['text'])
        })
    
    df_chunks = pd.DataFrame(chunks_data)
    print(f"Found {len(df_chunks)} chunks for document {document_id}\n")
    df_chunks[['Index', 'Content Preview', 'Page', 'Length']]
    
except Exception as e:
    print(f"Error: {e}")
    print("Make sure the document ID is correct and the document has been indexed.")

In [15]:
# Select a document ID
document_id = "8269b380-d311-472e-b5ed-d3e4476ba880"  # Replace with your document ID

# Initialize vector store
vector_store = ChromaVectorStore()

# Get chunks for this document
try:
    collection = vector_store._get_collection(document_id)
    
    # Get all chunks
    results = collection.get(
        include=["documents", "metadatas"]
    )
    
    chunks_data = []
    for i, (chunk_id, content, metadata) in enumerate(zip(results['ids'], results['documents'], results['metadatas'])):
        chunks_data.append({
            'Index': i,
            'Chunk ID': chunk_id,
            'Content Preview': content[:100] + '...' if len(content) > 100 else content,
            'Full Content': content,
            'Page': metadata.get('page_number', 'N/A'),
            'Length': len(content)
        })
    
    df_chunks = pd.DataFrame(chunks_data)
    print(f"Found {len(df_chunks)} chunks for document {document_id}\n")
    df_chunks[['Index', 'Content Preview', 'Page', 'Length']]
    
except Exception as e:
    print(f"Error: {e}")
    print("Make sure the document ID is correct and the document has been indexed.")

Error: 'ChromaVectorStore' object has no attribute '_get_collection'
Make sure the document ID is correct and the document has been indexed.


## 3. View Full Content of a Specific Chunk

In [ ]:
# Select a chunk index from the table above
chunk_index = 0

if chunk_index < len(df_chunks):
    chunk = df_chunks.iloc[chunk_index]
    print(f"Chunk {chunk_index}:")
    print(f"Page: {chunk['Page']}")
    print(f"Length: {chunk['Length']} characters\n")
    print("Full Content:")
    print("=" * 80)
    print(chunk['Full Content'])
    print("=" * 80)
else:
    print(f"Chunk index {chunk_index} not found. Available indices: 0-{len(df_chunks)-1}")

## 4. Search for Chunks

Test the contextual retrieval search:

In [ ]:
# Initialize the retrieval service
retrieval_service = ContextualRetrievalService()

# Search query
query = "coverage limits"  # Change this to your search query
top_k = 5  # Number of results to return

# Perform search (optionally filter by document)
results = retrieval_service.search(
    query=query,
    top_k=top_k,
    filter_doc_id=document_id  # Remove this line to search across all documents
)

print(f"Search results for: '{query}'\n")
print(f"Found {len(results)} results:\n")

for i, result in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"  Score: {result.score:.4f}")
    print(f"  Document ID: {result.document_id}")
    print(f"  Chunk ID: {result.chunk_id}")
    print(f"  Content: {result.content[:200]}...")
    print()

## 5. Analyze Chunk Statistics

In [ ]:
import matplotlib.pyplot as plt

if len(df_chunks) > 0:
    # Chunk length distribution
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.hist(df_chunks['Length'], bins=20, edgecolor='black')
    plt.xlabel('Chunk Length (characters)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Chunk Lengths')
    plt.grid(axis='y', alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.boxplot(df_chunks['Length'])
    plt.ylabel('Chunk Length (characters)')
    plt.title('Chunk Length Box Plot')
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nChunk Statistics:")
    print(f"  Total chunks: {len(df_chunks)}")
    print(f"  Average length: {df_chunks['Length'].mean():.0f} characters")
    print(f"  Median length: {df_chunks['Length'].median():.0f} characters")
    print(f"  Min length: {df_chunks['Length'].min()} characters")
    print(f"  Max length: {df_chunks['Length'].max()} characters")
else:
    print("No chunks to analyze. Select a document first.")

## 6. Compare Search Methods

Compare vector search, BM25, and hybrid search:

In [ ]:
query = "insurance coverage"  # Change this to your query

# Get retriever
retriever = retrieval_service.retriever

# Vector-only search
vector_results = retriever.retrieve_vector_only(query, top_k=5, filter_doc_id=document_id)

# BM25-only search
bm25_results = retriever.retrieve_bm25_only(query, top_k=5, filter_doc_id=document_id)

# Hybrid search (with reranking)
hybrid_results = retriever.retrieve(query, top_k_final=5, filter_doc_id=document_id, use_reranking=True)

print(f"Search comparison for: '{query}'\n")
print("=" * 80)

print("\n1. VECTOR SEARCH (Semantic):")
for i, result in enumerate(vector_results, 1):
    print(f"  {i}. Score: {result.score:.4f} | {result.content[:80]}...")

print("\n2. BM25 SEARCH (Keyword):")
for i, result in enumerate(bm25_results, 1):
    print(f"  {i}. Score: {result.score:.4f} | {result.content[:80]}...")

print("\n3. HYBRID SEARCH (Vector + BM25 + Reranking):")
for i, result in enumerate(hybrid_results, 1):
    print(f"  {i}. Score: {result.score:.4f} | {result.content[:80]}...")